## AgenticAI
1. AI systems that can work foryouindependently
2. A system in which an LLM controls the workflow
3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition.
Let's understand this shiet

In [1]:
from typing import override
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override = True)

True

In [2]:
def show(text):
    try: 
        Console().print(text)
    except:
        print(text)
        

In [3]:
openai = OpenAI()

In [4]:
# The list we will be using
todos = []
completed = []

In [14]:
def get_todo_report() ->str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index+1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index+1}: {todo} \n"
    show(result)
    return result

In [15]:
get_todo_report()

Todo #1: prenderlo
Todo #2: Buy groceries 
Todo #3: Piano

'Todo #1: [green][strike]prenderlo[/strike][/green]\nTodo #2: Buy groceries \nTodo #3: Piano \n'

In [17]:
def create_todos(descriptions: list[str])->str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [22]:
def mark_complete(index: int, completion_notes: str) ->str:
    if 1<= index <= len(todos):
        completed[index - 1] = True
    else:
        return "Not todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [19]:
## Testing thefunctions
todos, completed = [],[]
create_todos(["prenderlo", "Buy groceries", "Piano"])

Todo #1: prenderlo 
Todo #2: Buy groceries 
Todo #3: Piano

'Todo #1: prenderlo \nTodo #2: Buy groceries \nTodo #3: Piano \n'

In [23]:
mark_complete(2,"suavelandi")

suavelandi

Todo #1: prenderlo
Todo #2: Buy groceries
Todo #3: Piano

'Todo #1: [green][strike]prenderlo[/strike][/green]\nTodo #2: [green][strike]Buy groceries[/strike][/green]\nTodo #3: Piano \n'

## JSON for tools

In [26]:
create_todos_JSON = {
    "name":"create_todos",
    "description":"Add new todos from a list of descriptions and return the full list",
    "parameters":{
        "type": "object",
        "properties":{
            "descriptions":{
                'type': 'array',
                'items': {'type':'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [27]:
mark_complete_JSON = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters":{
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title':'Index',
                'type':'integer',
                },
            'completion_notes':{
                'description':'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index','completion_notes'],
        'type': 'object',
        'additionalProperties':False
    }
}

In [29]:
tools = [{"type": "function", "function":create_todos_JSON},
         {"type": "function","function":mark_complete_JSON}]

In [31]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role":"tool","content":json.dumps(result), "tool_call_id":tool_call.id})
    return results

In [32]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model = "gpt-5.2", messages = messages, tools = tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason == "tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0]. message.content)

In [33]:
system_message = """
You are given a problem to solve by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to compe up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks. 
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
when do they meet? 
"""
messages = [{"role":"system","content":system_message}, 
            {"role": "user", "content": user_message}]

In [35]:
todos, completed = [], []
loop(messages)

Todo #1: Parse the problem and define variables, including departure times, speeds, and initial separation between 
Boston and New York. 
Todo #2: Estimate/choose a reasonable distance between Boston and New York (since not provided). 
Todo #3: Set up relative-motion equation accounting for the 1-hour head start and solve for meeting time. 
Todo #4: Compute the clock time of meeting and present clearly.

Let Boston–New York distance be D miles. Train A leaves Boston 2:00 pm at 60 mph toward NY. Train B leaves NY 3:00 
pm at 80 mph toward Boston. Train A has a 1-hour head start, so by 3:00 pm it has gone 60 miles, leaving (D−60) 
miles between them at 3:00 pm.

Todo #1: Parse the problem and define variables, including departure times, speeds, and initial separation between 
Boston and New York.
Todo #2: Estimate/choose a reasonable distance between Boston and New York (since not provided). 
Todo #3: Set up relative-motion equation accounting for the 1-hour head start and solve for meeting time. 
Todo #4: Compute the clock time of meeting and present clearly.

Distance not given; use a reasonable real-world estimate. Typical rail/road distance Boston↔NYC is about 215 miles 
(order of 200–220). I’ll use D = 215 miles.

Todo #1: Parse the problem and define variables, including departure times, speeds, and initial separation between 
Boston and New York.
Todo #2: Estimate/choose a reasonable distance between Boston and New York (since not provided).
Todo #3: Set up relative-motion equation accounting for the 1-hour head start and solve for meeting time. 
Todo #4: Compute the clock time of meeting and present clearly.

At 3:00 pm, separation is D−60 = 215−60 = 155 miles. From 3:00 pm onward they approach each other at relative speed
60+80 = 140 mph. Time after 3:00 pm to meet: t = 155/140 = 1.107142… hr ≈ 1 hr 6 min 26 s.

Todo #1: Parse the problem and define variables, including departure times, speeds, and initial separation between 
Boston and New York.
Todo #2: Estimate/choose a reasonable distance between Boston and New York (since not provided).
Todo #3: Set up relative-motion equation accounting for the 1-hour head start and solve for meeting time.
Todo #4: Compute the clock time of meeting and present clearly.

Meeting time = 3:00 pm + 1 hr 6 min 26 s ≈ 4:06 pm (about 4:06:26 pm).

Todo #1: Parse the problem and define variables, including departure times, speeds, and initial separation between 
Boston and New York.
Todo #2: Estimate/choose a reasonable distance between Boston and New York (since not provided).
Todo #3: Set up relative-motion equation accounting for the 1-hour head start and solve for meeting time.
Todo #4: Compute the clock time of meeting and present clearly.

Assuming Boston–New York is about 215 miles apart (a common real-world estimate):

- By 3:00 pm, the Boston train (60 mph) has a 1-hour head start → it travels 60 miles.
- Remaining distance between trains at 3:00 pm: 215 − 60 = 155 miles.
- From 3:00 pm on, they approach each other at 60 + 80 = 140 mph.
- Time to meet after 3:00 pm:  
  155 / 140 = 1.1071 hours ≈ 1 hour 6 minutes 26 seconds.

They meet at about 4:06 pm (approximately 4:06:26 pm).